In [ ]:
!pip install qiskit qiskit-aer pandas numpy matplotlib --quiet
!pip install pylatexenc

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error
from qiskit_aer import AerSimulator
from qiskit import transpile
import pandas as pd
import numpy as n

In [ ]:
# 1. Build the distance-3 surface code circuit
def build_surface_code_circuit(rounds=5):
    """
    Constructs a distance-3 rotated surface code circuit.
    Uses 9 data qubits and 8 ancilla qubits.
    """
    data = QuantumRegister(9, name='data')
    ancilla = QuantumRegister(8, name='ancilla')

    # Classical registers: one for each round (8 bits each), plus final data
    c_syndrome = [ClassicalRegister(8, name=f'round_{r}') for r in range(rounds)]
    c_final = ClassicalRegister(9, name='final_data')

    qc = QuantumCircuit(data, ancilla, *c_syndrome, c_final)

    # Distance-3 stabilizer map: (ancilla_index, [data_qubits])
    stabilizer_map = [
        (0, [0,1,3,4]),   # X-type (even) on top-left plaquette
        (2, [1,2,4,5]),   # X-type on top-right
        (4, [3,4,6,7]),   # X-type on bottom-left
        (6, [4,5,7,8]),   # X-type on bottom-right
        (1, [0,3]),       # Z-type (odd) left edge
        (3, [1,2,4,5]),   # Z-type middle vertical
        (5, [3,4,6,7]),   # Z-type middle horizontal
        (7, [5,8])        # Z-type right edge
    ]

    for r in range(rounds):
        # Reset ancillas
        qc.reset(ancilla)

        # Prepare X-stabilizers (Hadamard on even-index ancillas)
        for a_idx, _ in stabilizer_map:
            if a_idx % 2 == 0:
                qc.h(ancilla[a_idx])

        # CNOT interactions
        for a_idx, neighbors in stabilizer_map:
            for q_idx in neighbors:
                if a_idx % 2 == 0:
                    qc.cx(ancilla[a_idx], data[q_idx])
                else:
                    qc.cx(data[q_idx], ancilla[a_idx])

        # Close X-stabilizers (Hadamard again)
        for a_idx, _ in stabilizer_map:
            if a_idx % 2 == 0:
                qc.h(ancilla[a_idx])

        # Measure ancillas
        for a_idx in range(8):
            qc.measure(ancilla[a_idx], c_syndrome[r][a_idx])

        qc.barrier()

    # Final data measurement
    for q_idx in range(9):
        qc.measure(data[q_idx], c_final[q_idx])

    return qc

In [ ]:
# 2. Noise models
def get_depolarizing_noise_model(gate_error=0.01):
    noise = NoiseModel()

    error_1q = depolarizing_error(gate_error, 1)
    error_2q = depolarizing_error(gate_error, 2)

    noise.add_all_qubit_quantum_error(error_1q, ['h'])
    noise.add_all_qubit_quantum_error(error_2q, ['cx'])
    return noise

def get_biased_noise_model(gate_error=0.01, meas_error=0.015, reset_error=0.005):
    noise = get_depolarizing_noise_model(gate_error)

    error_meas = pauli_error([('X', meas_error), ('I', 1 - meas_error)])
    error_reset = pauli_error([('X', reset_error), ('I', 1 - reset_error)])

    noise.add_all_qubit_quantum_error(error_meas, ['measure'])
    noise.add_all_qubit_quantum_error(error_reset, ['reset'])
    return noise

In [ ]:
# 3. Simulation and extraction
def simulate_and_extract(qc, noise_model, shots):
    """
    Runs simulation and extracts features (40-bit syndromes) and labels.
    """
    sim = AerSimulator(method='stabilizer')  # fast for large shots
    qc_transpiled = transpile(qc, sim)
    result = sim.run(qc_transpiled, noise_model=noise_model, shots=shots, memory=True).result()
    memory = result.get_memory()

    features = []
    labels = []

    for shot_string in memory:
        parts = shot_string.split()
        # parts[0] is final data, rest are rounds in reverse order (round_4 ... round_0)
        syndrome_bits = []
        for round_str in reversed(parts[1:]):
            # Each round string has 8 bits, but order is reversed: we want ancilla0 first
            syndrome_bits.extend([int(b) for b in reversed(round_str)])
        features.append(syndrome_bits)

        # Logical label: XOR of top boundary data qubits 0,1,2
        final_data_str = parts[0]  # 9 bits, last character is data qubit 0
        q0 = int(final_data_str[-1])
        q1 = int(final_data_str[-2])
        q2 = int(final_data_str[-3])
        logical_flip = q0 ^ q1 ^ q2
        labels.append(logical_flip)

    return features, labels

In [ ]:
# 4. Generate datasets
NUM_SHOTS = 10000

print("Generating distance-3 circuit...")
qc = build_surface_code_circuit(rounds=5)

# Dataset 1: Symmetric depolarising noise
print(f"Simulating {NUM_SHOTS} shots with depolarising noise...")
noise_depol = get_depolarizing_noise_model()
feat_depol, lab_depol = simulate_and_extract(qc, noise_depol, NUM_SHOTS)

pd.DataFrame(feat_depol).to_csv('features_pretrain.csv', index=False)
pd.DataFrame(lab_depol, columns=['label']).to_csv('labels_pretrain.csv', index=False)

# Dataset 2: Biased noise
print(f"Simulating {NUM_SHOTS} shots with biased noise...")
noise_biased = get_biased_noise_model()
feat_biased, lab_biased = simulate_and_extract(qc, noise_biased, NUM_SHOTS)

pd.DataFrame(feat_biased).to_csv('features_finetune.csv', index=False)
pd.DataFrame(lab_biased, columns=['label']).to_csv('labels_finetune.csv', index=False)

print("\nData generation complete!")
print("Features shape (symmetric):", pd.read_csv('features_pretrain.csv').shape)
print("Labels shape (symmetric):", pd.read_csv('labels_pretrain.csv').shape)
print("Features shape (biased):", pd.read_csv('features_finetune.csv').shape)
print("Labels shape (biased):", pd.read_csv('labels_finetune.csv').shape)

# Optional: quick preview
print("\nSymmetric label distribution:")
print(pd.read_csv('labels_pretrain.csv')['label'].value_counts())